In [16]:
from datasets import load_dataset
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

In [17]:
from datasets import IterableDataset

dataset = dataset.take(100_000)

In [18]:
with open('data.txt', 'w') as file:
  for i, example in enumerate(dataset):
    file.write(example['text'] + '\n')

In [19]:
import pandas as pd

df = pd.DataFrame(dataset)

del dataset

In [20]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

tokenizer = Tokenizer(BPE())

tokenizer.pre_tokenizer = ByteLevel()

trainer = BpeTrainer(
    vocab_size=50257,
    special_tokens=["<|endoftext|>"],
    initial_alphabet=ByteLevel.alphabet()
)

tokenizer.decoder = ByteLevelDecoder()

tokenizer.train(
    files=["data.txt"],
    trainer=trainer
)

tokenizer.save("tokenizer.json")

In [21]:
import torch
from torch.utils.data import Dataset

class GPT2Dataset(Dataset):
    def __init__(self, df, tokenizer, block_size=512):
        super().__init__()
        self.block_size = block_size
        self.tokenizer = tokenizer
        self.df = df
        self.pad_id = tokenizer.token_to_id('<|endoftext|>')

    def __pad(self, ids):
        ids = ids[:self.block_size]
        ids = ids + [self.pad_id] * (self.block_size - len(ids))
        return ids

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df['text'][idx]
        encoding = self.tokenizer.encode(text)
        ids = self.__pad(encoding.ids)

        input_ids = torch.tensor(ids[:-1], dtype=torch.long)
        target_ids = torch.tensor(ids[1:], dtype=torch.long)
        target_ids[target_ids == self.pad_id] = -100

        return input_ids, target_ids

In [22]:
VOCAB_SIZE = tokenizer.get_vocab_size()

In [23]:
import torch
import torch.nn as nn


class RoPE(nn.Module):
    def __init__(self, d_model, max_len=512, base=10000):
        super().__init__()

        # Frequencies for each pair of dimensions
        inv_freq = 1.0 / (
            base ** (torch.arange(0, d_model, 2).float() / d_model)
        )

        # Positions: [0, 1, 2, ..., max_len-1]
        positions = torch.arange(max_len).float()

        # [max_len, d_model / 2]
        freqs = torch.outer(positions, inv_freq)

        # Precompute
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    def forward(self, x):
        """
        x shape:
            [..., seq_len, d_model]  (works for [batch, seq_len, d_model]
            or [batch, num_heads, seq_len, head_dim])
        """

        seq_len = x.size(-2)

        cos = self.cos[:seq_len]
        sin = self.sin[:seq_len]

        # Split even and odd dimensions
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        # Apply rotation
        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos

        # Put dimensions back together
        x_rotated = torch.stack(
            [rotated_even, rotated_odd],
            dim=-1
        )

        x_rotated = x_rotated.flatten(-2)

        return x_rotated

In [24]:
import torch.nn as nn
import torch

class embeddingGPT(nn.Module):
  def __init__(self, d_model=768, max_len=512, vocab_size=VOCAB_SIZE):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
    self.max_len = max_len  # kept for interface compat / used by RoPE downstream

  def forward(self, x):
    return self.embedding(x)


In [25]:
class feedforward(nn.Module):
  def __init__(self, d_model, d_ff=None, dropout=0.1):
    super().__init__()
    d_ff = d_ff or d_model * 4
    self.linear_layer1 = nn.Linear(in_features=d_model, out_features=d_ff)
    self.Gelu = nn.GELU()
    self.linear_layer2 = nn.Linear(in_features=d_ff, out_features=d_model)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.dropout(self.linear_layer2(self.Gelu(self.linear_layer1(x))))

In [26]:
import torch.nn.functional as F

class RoPEMultiHeadAttention(nn.Module):
  """
  Multi-head self-attention with RoPE applied to q and k.

  RoPE has to rotate q/k AFTER they've been split into per-head vectors,
  not on the raw d_model input before projection -- rotating x and then
  projecting with W_q is not the same as projecting then rotating, so
  nn.MultiheadAttention (which hides the q/k projection inside itself)
  can't be used here. This class exposes q_proj/k_proj/v_proj explicitly
  so RoPE can be inserted at the right point, on head_dim-sized vectors.
  """
  def __init__(self, d_model, num_heads, max_len=512, base=10000, dropout=0.1):
    super().__init__()
    assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

    self.num_heads = num_heads
    self.head_dim = d_model // num_heads

    self.q_proj = nn.Linear(d_model, d_model)
    self.k_proj = nn.Linear(d_model, d_model)
    self.v_proj = nn.Linear(d_model, d_model)
    self.out_proj = nn.Linear(d_model, d_model)

    self.dropout_p = dropout

    # RoPE is built on head_dim (e.g. 64), not d_model (e.g. 1024) --
    # each head gets its own independent rotation.
    self.rope = RoPE(self.head_dim, max_len=max_len, base=base)

  def forward(self, x, attn_mask=None):
    batch, seq_len, d_model = x.shape

    q = self.q_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
    # q, k, v: [batch, num_heads, seq_len, head_dim]

    q = self.rope(q)
    k = self.rope(k)

    attn_out = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=attn_mask,
        dropout_p=self.dropout_p if self.training else 0.0,
    )
    # attn_out: [batch, num_heads, seq_len, head_dim]

    attn_out = attn_out.transpose(1, 2).contiguous().view(batch, seq_len, d_model)
    return self.out_proj(attn_out)


class Transformer(nn.Module):
  def __init__(self, d_model, num_heads, max_len=512, dropout=0.1):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(d_model)
    self.masked_attn = RoPEMultiHeadAttention(
        d_model=d_model, num_heads=num_heads, max_len=max_len, dropout=dropout
    )

    self.layer_norm2 = nn.LayerNorm(d_model)
    self.feedforward = feedforward(d_model, dropout=dropout)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x, attn_mask):
    normed = self.layer_norm1(x)
    attn_out = self.masked_attn(normed, attn_mask=attn_mask)
    x = x + self.dropout(attn_out)

    normed2 = self.layer_norm2(x)
    ff_out = self.feedforward(normed2)
    x = x + ff_out

    return x


In [27]:
import torch.nn as nn
import torch

class GPT2(nn.Module):
  def __init__(self, d_model, num_heads, vocab_size=VOCAB_SIZE, max_len=512, num_layers=12, dropout=0.1):
    super().__init__()

    self.embedding = embeddingGPT(d_model=d_model, max_len=max_len, vocab_size=vocab_size)

    self.blocks = nn.ModuleList([
                    Transformer(
                        d_model=d_model,
                        num_heads=num_heads,
                        max_len=max_len,
                        dropout=dropout
                    )
                    for _ in range(num_layers)
                ])

    self.ln_f = nn.LayerNorm(d_model)
    self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
    self.max_len = max_len

  def forward(self, x, attn_mask=None):
    seq_len = x.size(1)

    if attn_mask is None:
      attn_mask = torch.triu(
          torch.full((seq_len, seq_len), float('-inf'), device=x.device),
          diagonal=1
      )

    x = self.embedding(x)

    for block in self.blocks:
      x = block(x, attn_mask)

    x = self.ln_f(x)
    logits = self.lm_head(x)

    return logits


In [28]:
from torch.utils.data import DataLoader, random_split

dataset = GPT2Dataset(df, tokenizer)

length = len(dataset)

train_size = int(length * 0.8)
val_size = length - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

In [29]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GPT2(d_model=1024, num_heads=16, vocab_size=VOCAB_SIZE).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1
)

In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for step, (input_ids, target_ids) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        optimizer.zero_grad()

        logits = model(input_ids) # (batch, seq_len, vocab_size)
        loss = loss_fn(
            logits.view(-1, logits.size(-1)),
            target_ids.view(-1)
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

        if step % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for input_ids, target_ids in val_loader:
            input_ids = input_ids.to(device)
            target_ids = target_ids.to(device)

            logits = model(input_ids)
            loss = loss_fn(
                logits.view(-1, logits.size(-1)),
                target_ids.view(-1)
            )
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"=== Epoch {epoch+1} done | Train loss {avg_train_loss:.4f} | Val loss {avg_val_loss:.4f} ===")

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=1.0, top_k=50, eos_token="<|endoftext|>"):
    model.eval()

    eos_id = tokenizer.token_to_id(eos_token)
    input_ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)  # (1, seq_len)

    for _ in range(max_new_tokens):
        context = input_ids[:, -model.max_len:]

        logits = model(context)
        next_token_logits = logits[:, -1, :] / temperature


        if top_k is not None:
            values, _ = torch.topk(next_token_logits, top_k)
            min_value = values[:, -1].unsqueeze(-1)
            next_token_logits = torch.where(
                next_token_logits < min_value,
                torch.full_like(next_token_logits, float('-inf')),
                next_token_logits
            )

        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

        if next_token.item() == eos_id:
            break

    generated_ids = input_ids[0].tolist()
    return tokenizer.decode(generated_ids)


output = generate("Once upon a time", max_new_tokens=100, temperature=0.8, top_k=50)
print(output)